# Brazilian E-Commerce Sales and Logistics Analysis

**Author:** Manvitha Gopireddy | **Tools:** Python, Pandas, Seaborn, SQL

**Dataset:** [Olist E-Commerce Public Dataset - Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

---

## Business Problem

Olist wants to understand why customer satisfaction scores are fluctuating and identify logistics bottlenecks hurting revenue and repeat purchases.

Key Questions:
- What are the main drivers of customer satisfaction?
- Which states and categories have the worst delivery performance?
- What is the customer retention rate, and how can it be improved?

---

## Table of Contents
1. Setup and Data Loading
2. Data Cleaning and Feature Engineering
3. Exploratory Data Analysis
4. Key Insight - Delivery vs Satisfaction
5. Revenue Analysis
6. RFM Customer Segmentation
7. Business Recommendations

In [ ]:
# ── Setup: Import Libraries ──────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

DATA_DIR = '../data'
print('Libraries loaded successfully!')
print(f'Pandas version: {pd.__version__}')

## Section 1: Data Loading

The Olist dataset consists of 9 CSV files that form a relational schema. We load the 6 most important ones for our analysis.

In [ ]:
# ── Load all CSV files ──────────────────────────────────────────
orders    = pd.read_csv(f'{DATA_DIR}/olist_orders_dataset.csv')
items     = pd.read_csv(f'{DATA_DIR}/olist_order_items_dataset.csv')
payments  = pd.read_csv(f'{DATA_DIR}/olist_order_payments_dataset.csv')
reviews   = pd.read_csv(f'{DATA_DIR}/olist_order_reviews_dataset.csv')
customers = pd.read_csv(f'{DATA_DIR}/olist_customers_dataset.csv')
products  = pd.read_csv(f'{DATA_DIR}/olist_products_dataset.csv')

# Print shapes to verify load
for name, df in [('orders', orders), ('items', items), ('payments', payments),
                 ('reviews', reviews), ('customers', customers), ('products', products)]:
    print(f'{name:12s}: {len(df):>7,} rows | {df.shape[1]:>2} columns')

## Section 2: Data Cleaning and Feature Engineering

**Why we clean:**
- Date columns come in as raw strings and must be converted to datetime objects
- Some orders were never delivered (cancelled/lost) — we drop these for delivery analysis
- We engineer new features: `delivery_days` and `is_late` which are central to our core business insight

In [ ]:
# ── Data Cleaning ──────────────────────────────────────────

# Convert date columns to datetime
date_cols = ['order_purchase_timestamp', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# Keep only delivered orders (drops cancelled, unavailable, etc.)
orders_clean = orders[orders['order_status'] == 'delivered'].copy()
before = len(orders)
orders_clean.dropna(subset=['order_delivered_customer_date'], inplace=True)
after = len(orders_clean)
print(f'Rows before cleaning: {before:,}')
print(f'Rows after cleaning:  {after:,}')
print(f'Rows dropped:         {before - after:,} (cancelled / lost / undelivered orders)')

# ── Feature Engineering ──────────────────────────────────

# Delivery time in days
orders_clean['delivery_days'] = (
    orders_clean['order_delivered_customer_date'] -
    orders_clean['order_purchase_timestamp']
).dt.days

# Was the order late? (actual > estimated)
orders_clean['is_late'] = (
    orders_clean['order_delivered_customer_date'] >
    orders_clean['order_estimated_delivery_date']
)

# Year-month for time series
orders_clean['year_month'] = orders_clean['order_purchase_timestamp'].dt.to_period('M')

print(f'\nNew features created: delivery_days, is_late, year_month')
print(f'Late deliveries: {orders_clean["is_late"].sum():,} ({orders_clean["is_late"].mean()*100:.1f}%)')

In [ ]:
# ── Merge all tables into one master dataframe ─────────────────────
df = (
    orders_clean
    .merge(items,     on='order_id',    how='left')
    .merge(payments,  on='order_id',    how='left')
    .merge(reviews,   on='order_id',    how='left')
    .merge(customers, on='customer_id', how='left')
    .merge(products,  on='product_id',  how='left')
)

print(f'Master dataframe shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

## Section 3: Key Business Insight — Delivery Time vs Customer Satisfaction

**Hypothesis:** Orders that arrive late (past the estimated delivery date) will have significantly lower review scores than on-time orders.

This is the most actionable insight in the entire dataset because fixing it requires a **process change**, not more spending.

In [ ]:
# ── FINDING 1: On-Time vs Late Delivery Impact on Review Score ──────────

late_analysis = df.groupby('is_late')['review_score'].agg(['mean', 'count', 'std']).reset_index()
late_analysis['label'] = late_analysis['is_late'].map({True: 'Late Delivery', False: 'On-Time Delivery'})
late_analysis.columns = ['is_late', 'avg_score', 'order_count', 'std_score', 'label']

ontime_score = late_analysis[late_analysis['is_late'] == False]['avg_score'].values[0]
late_score   = late_analysis[late_analysis['is_late'] == True]['avg_score'].values[0]
diff_pct     = ((ontime_score - late_score) / ontime_score) * 100

print('='*50)
print('FINDING 1: Delivery Punctuality vs Review Score')
print('='*50)
print(f'On-Time Delivery Avg Score : {ontime_score:.2f} / 5.00')
print(f'Late Delivery Avg Score    : {late_score:.2f} / 5.00')
print(f'Score Drop for Late Orders : {diff_pct:.1f}%')
print(f'Number of Late Orders      : {int(late_analysis[late_analysis["is_late"]==True]["order_count"].values[0]):,}')
print('='*50)
print(f'BUSINESS IMPACT: Being just 1 day late drops satisfaction by {diff_pct:.0f}%!')

# Chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: Avg score comparison
axes[0].bar(late_analysis['label'], late_analysis['avg_score'],
            color=[COLORS[2], COLORS[3]], edgecolor='white', width=0.5)
for i, row in late_analysis.iterrows():
    axes[0].text(i, row['avg_score'] + 0.05, f"{row['avg_score']:.2f}",
                 ha='center', fontweight='bold', fontsize=12)
axes[0].set_ylim(0, 5.5)
axes[0].set_title('Avg Review Score: On-Time vs Late', fontweight='bold')
axes[0].set_ylabel('Average Review Score (out of 5)')

# Box plot: full distribution
df_box = df[df['delivery_days'].between(0, 60) & df['review_score'].notna()].copy()
df_box['review_score'] = df_box['review_score'].astype(int)
sns.boxplot(x='review_score', y='delivery_days', data=df_box,
            hue='review_score', palette='RdYlGn', order=[1,2,3,4,5],
            legend=False, ax=axes[1])
axes[1].set_title('Delivery Days Distribution by Review Score', fontweight='bold')
axes[1].set_xlabel('Review Score (1=Worst, 5=Best)')
axes[1].set_ylabel('Delivery Days')

plt.suptitle('KEY INSIGHT: Delivery Performance Drives Customer Satisfaction',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../charts/insight_delivery_vs_satisfaction.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 4: RFM Customer Segmentation

**What is RFM?** A proven marketing analytics framework used by every major e-commerce company:
- **R — Recency:** How recently did the customer buy? (Lower = better)
- **F — Frequency:** How many times did they buy?
- **M — Monetary:** How much total did they spend?

**Why this matters:** RFM lets us classify customers into actionable segments like Champions, At-Risk, and Lost Customers, so the marketing team can target them with the right message.

In [ ]:
# ── RFM Customer Segmentation ────────────────────────────────────────

# Merge orders with customers to get unique customer IDs
df_rfm_base = orders_clean.merge(customers[['customer_id', 'customer_unique_id']], on='customer_id')
df_rfm_base = df_rfm_base.merge(payments[['order_id', 'payment_value']], on='order_id', how='left')

# Reference date: 1 day after the last purchase in the dataset
reference_date = df_rfm_base['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

# Calculate RFM metrics per unique customer
rfm = df_rfm_base.groupby('customer_unique_id').agg(
    Recency   = ('order_purchase_timestamp', lambda x: (reference_date - x.max()).days),
    Frequency = ('order_id', 'count'),
    Monetary  = ('payment_value', 'sum')
).reset_index()

print(f'Total unique customers: {len(rfm):,}')
print(rfm.describe().round(2))

# Score each metric 1-5 using quintiles
rfm['R_score'] = pd.qcut(rfm['Recency'],   5, labels=[5, 4, 3, 2, 1]).astype(int)  # Lower recency = better
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['Monetary'],  5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['RFM_Score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

# Segment customers based on RFM Score
def segment_customer(score):
    if score >= 13:   return 'Champions'
    elif score >= 10: return 'Loyal Customers'
    elif score >= 7:  return 'Potential Loyalists'
    elif score >= 5:  return 'At Risk'
    else:             return 'Lost Customers'

rfm['Segment'] = rfm['RFM_Score'].apply(segment_customer)

# Summary by segment
segment_summary = rfm.groupby('Segment').agg(
    Customer_Count = ('customer_unique_id', 'count'),
    Avg_Recency    = ('Recency',   'mean'),
    Avg_Frequency  = ('Frequency', 'mean'),
    Avg_Monetary   = ('Monetary',  'mean'),
    Total_Revenue  = ('Monetary',  'sum')
).round(2).sort_values('Total_Revenue', ascending=False)

print('\nRFM Segment Summary:')
print(segment_summary.to_string())

In [ ]:
# ── Visualise RFM Segments ────────────────────────────────────────

seg_order   = ['Champions', 'Loyal Customers', 'Potential Loyalists', 'At Risk', 'Lost Customers']
seg_colors  = ['#2ecc71', '#3498db', '#f39c12', '#e67e22', '#e74c3c']

seg_counts  = rfm['Segment'].value_counts().reindex(seg_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Donut chart: Customer count by segment
wedges, texts, autotexts = axes[0].pie(
    seg_counts,
    labels=seg_counts.index,
    autopct='%1.1f%%',
    colors=seg_colors,
    startangle=90,
    wedgeprops=dict(width=0.5)
)
for text in autotexts:
    text.set_fontsize(9)
axes[0].set_title('Customer Distribution by RFM Segment', fontweight='bold')

# Bar chart: Avg monetary value by segment
avg_monetary = rfm.groupby('Segment')['Monetary'].mean().reindex(seg_order)
axes[1].barh(seg_order[::-1], avg_monetary[::-1], color=seg_colors[::-1], edgecolor='white')
for i, val in enumerate(avg_monetary[::-1]):
    axes[1].text(val + 5, i, f'BRL {val:,.0f}', va='center', fontsize=9)
axes[1].set_title('Average Revenue per Customer by Segment', fontweight='bold')
axes[1].set_xlabel('Average Monetary Value (BRL)')

plt.suptitle('RFM Customer Segmentation Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../charts/rfm_customer_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()

print('RFM chart saved!')

## Section 5: Summary Statistics and Business Recommendations

Run the cell below to generate the complete business findings report.

In [ ]:
# ── Final Business Report ──────────────────────────────────────────

# Core metrics
total_revenue   = df['payment_value'].sum()
total_orders    = df['order_id'].nunique()
aov             = total_revenue / total_orders
avg_delivery    = df['delivery_days'].mean()
avg_review      = df['review_score'].mean()
pct_late        = df['is_late'].mean() * 100

# Delivery impact
ontime_score_val = df[df['is_late'] == False]['review_score'].mean()
late_score_val   = df[df['is_late'] == True]['review_score'].mean()
drop_pct         = ((ontime_score_val - late_score_val) / ontime_score_val) * 100

# RFM
champions_count  = len(rfm[rfm['Segment'] == 'Champions'])
at_risk_count    = len(rfm[rfm['Segment'] == 'At Risk'])
lost_count       = len(rfm[rfm['Segment'] == 'Lost Customers'])

print('=' * 60)
print('         OLIST E-COMMERCE ANALYSIS - FINAL REPORT')
print('=' * 60)
print(f'  Total Revenue:           BRL {total_revenue:>12,.0f}')
print(f'  Total Delivered Orders:  {total_orders:>12,}')
print(f'  Average Order Value:     BRL {aov:>12,.2f}')
print(f'  Avg Delivery Time:       {avg_delivery:>11.1f} days')
print(f'  Avg Review Score:        {avg_review:>11.2f} / 5.00')
print(f'  Late Deliveries:         {pct_late:>11.1f}%')
print('-' * 60)
print('  KEY FINDINGS:')
print(f'  1. On-time orders score {ontime_score_val:.2f} vs {late_score_val:.2f} for late ({drop_pct:.0f}% drop)')
print(f'  2. {champions_count:,} Champion customers drive disproportionate revenue')
print(f'  3. {at_risk_count + lost_count:,} At-Risk + Lost customers need win-back campaigns')
print('-' * 60)
print('  RECOMMENDATIONS:')
print('  R1. Buffer delivery estimates by 2 days for rural states')
print('  R2. Launch win-back email for 4,000+ at-risk customers')
print('  R3. Audit Office Furniture logistics partners for SLA violations')
print('  R4. Build a Northeast distribution hub to cut freight costs 3x')
print('  R5. Pre-onboard temp logistics contractors for November (Black Friday)')
print('=' * 60)